# Apache Hudi - Production Best Practices & Monitoring

Running Apache Hudi in production requires more than writing data successfully. You also need reliable Spark configuration, metadata indexing, table maintenance, observability, and debugging patterns.

## What you will learn

In this notebook, you will learn:

- Recommended Spark settings for Hudi workloads
- Enabling adaptive query execution and metadata indexing
- Creating a small production-style Hudi table for testing
- Checking query plans with `explain(True)`
- Inspecting Hudi metadata columns and commit activity
- Understanding cleaning, retention, and table maintenance concepts
- Debugging common production issues
- Practical scaling guidelines for Hudi on Spark

## Step 1 — Create an optimized Spark session

This Spark session uses common production-oriented settings.

The Hudi JAR is expected to be already available in the Docker image under `/opt/spark/jars`, so this notebook does not use `spark.jars.packages`.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Hudi-10-Production-Best-Practices")
    .master("spark://spark-master:7077")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.hudi.metadata.enabled", "true")
    .config("spark.sql.hudi.metadata.precise.skip", "true")
    .config("spark.sql.hudi.metadata.leaf.column.default", "true")
    .config("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark application id:", spark.sparkContext.applicationId)

Spark version: 4.0.2
Spark application id: app-20260425195539-0006


## Step 2 — Review important Spark configuration

These values influence performance, query planning, and Hudi metadata behavior.

In [2]:
important_configs = [
    "spark.serializer",
    "spark.sql.shuffle.partitions",
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.hudi.metadata.enabled",
    "spark.sql.hudi.metadata.precise.skip",
    "spark.sql.hudi.metadata.leaf.column.default",
    "spark.sql.extensions",
    "spark.sql.catalog.spark_catalog",
]

for key in important_configs:
    print(f"{key} = {spark.conf.get(key, '<not set>')}")

spark.serializer = org.apache.spark.serializer.KryoSerializer
spark.sql.shuffle.partitions = 200
spark.sql.adaptive.enabled = true
spark.sql.adaptive.coalescePartitions.enabled = true
spark.sql.hudi.metadata.enabled = true
spark.sql.hudi.metadata.precise.skip = true
spark.sql.hudi.metadata.leaf.column.default = true
spark.sql.extensions = org.apache.spark.sql.hudi.HoodieSparkSessionExtension
spark.sql.catalog.spark_catalog = org.apache.spark.sql.hudi.catalog.HoodieCatalog


## Step 3 — Define table paths

This notebook is independent and creates its own production-style demo table.

In [3]:
BASE_PATH = "/workspace/data/hudi"
TABLE_NAME = "riders_production_monitoring"
TABLE_PATH = f"{BASE_PATH}/{TABLE_NAME}"

print("Table name:", TABLE_NAME)
print("Table path:", TABLE_PATH)

Table name: riders_production_monitoring
Table path: /workspace/data/hudi/riders_production_monitoring


## Step 4 — Clean previous tutorial run

For a tutorial notebook, we start from a clean table path.

In production, never delete a table path manually unless this is an intentional destructive operation.

In [4]:
import os
import shutil

if os.path.exists(TABLE_PATH):
    shutil.rmtree(TABLE_PATH)
    print(f"Removed previous tutorial table: {TABLE_PATH}")
else:
    print("No previous table found.")

No previous table found.


## Step 5 — Create production-style sample data

The dataset includes multiple partitions so we can inspect partition pruning and query plans.

In [5]:
from pyspark.sql.functions import to_timestamp

df = spark.createDataFrame(
    [
        ("r1", "SFO", "2024-01-10 10:00:00", 4.9),
        ("r2", "SFO", "2024-01-10 10:05:00", 4.7),
        ("r3", "NYC", "2024-01-10 10:10:00", 4.5),
        ("r4", "LAX", "2024-01-10 10:15:00", 4.8),
        ("r5", "SEA", "2024-01-10 10:20:00", 4.3),
    ],
    ["rider", "city", "ts", "rating"]
).withColumn("ts", to_timestamp("ts"))

df.show(truncate=False)
df.printSchema()

[Stage 1:===================>                                       (1 + 2) / 3]

+-----+----+-------------------+------+
|rider|city|ts                 |rating|
+-----+----+-------------------+------+
|r1   |SFO |2024-01-10 10:00:00|4.9   |
|r2   |SFO |2024-01-10 10:05:00|4.7   |
|r3   |NYC |2024-01-10 10:10:00|4.5   |
|r4   |LAX |2024-01-10 10:15:00|4.8   |
|r5   |SEA |2024-01-10 10:20:00|4.3   |
+-----+----+-------------------+------+

root
 |-- rider: string (nullable = true)
 |-- city: string (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- rating: double (nullable = true)



## Step 6 — Write a Hudi table with production-oriented options

These options demonstrate common production concerns:

- Record keys and precombine fields
- Partitioning
- Metadata table
- Cleaner retention
- Small file handling basics

In [6]:
hudi_write_options = {
    "hoodie.table.name": TABLE_NAME,
    "hoodie.datasource.write.table.name": TABLE_NAME,
    "hoodie.datasource.write.recordkey.field": "rider",
    "hoodie.datasource.write.partitionpath.field": "city",
    "hoodie.datasource.write.precombine.field": "ts",
    "hoodie.datasource.write.table.type": "COPY_ON_WRITE",
    "hoodie.datasource.write.operation": "upsert",

    # Metadata table and file listing performance
    "hoodie.metadata.enable": "true",

    # Cleaning and retention demo settings
    "hoodie.clean.automatic": "true",
    "hoodie.clean.async": "false",
    "hoodie.cleaner.policy": "KEEP_LATEST_COMMITS",
    "hoodie.cleaner.commits.retained": "3",

    # Small file handling demo setting
    "hoodie.parquet.small.file.limit": str(64 * 1024 * 1024),
}

(
    df.write
    .format("hudi")
    .options(**hudi_write_options)
    .mode("overwrite")
    .save(TABLE_PATH)
)

print(f"Created Hudi table at: {TABLE_PATH}")

26/04/25 19:55:47 WARN ConfigUtils: The configuration key 'hoodie.cleaner.policy' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.policy' instead.
26/04/25 19:55:47 WARN ConfigUtils: The configuration key 'hoodie.cleaner.policy' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.policy' instead.
26/04/25 19:55:47 WARN ConfigUtils: The configuration key 'hoodie.cleaner.commits.retained' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.commits.retained' instead.
26/04/25 19:55:48 WARN ConfigUtils: The configuration key 'hoodie.cleaner.policy' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.policy' instead.
26/04/25 19:55:48 WARN ConfigUtils: The configuration key 'hoodie.cleaner.commits.retained' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.commits.retained' instead.
# WARNING:

26/04/25 19:56:04 WARN ConfigUtils: The configuration key 'hoodie.cleaner.policy' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.policy' instead.
26/04/25 19:56:04 WARN ConfigUtils: The configuration key 'hoodie.cleaner.commits.retained' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.commits.retained' instead.


[Stage 35:>                                                         (0 + 4) / 4]

26/04/25 19:56:09 WARN HoodieTableFileSystemView: Partition: SFO is not available in store
26/04/25 19:56:09 WARN HoodieTableFileSystemView: Partition: LAX is not available in store
26/04/25 19:56:09 WARN HoodieTableFileSystemView: Partition: NYC is not available in store
26/04/25 19:56:09 WARN HoodieTableFileSystemView: Partition: SEA is not available in store


26/04/25 19:56:13 WARN ConfigUtils: The configuration key 'hoodie.clean.async' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.async.enabled' instead.
26/04/25 19:56:14 WARN ConfigUtils: The configuration key 'hoodie.cleaner.policy' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.policy' instead.
26/04/25 19:56:14 WARN ConfigUtils: The configuration key 'hoodie.cleaner.commits.retained' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.commits.retained' instead.
26/04/25 19:56:14 WARN ConfigUtils: The configuration key 'hoodie.cleaner.policy' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.policy' instead.
26/04/25 19:56:14 WARN ConfigUtils: The configuration key 'hoodie.cleaner.policy' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.policy' instead.
26/04/25 19:56:14 WARN Con

## Step 7 — Register a temporary view

We use a temporary view for SQL examples in this notebook.

In [7]:
spark.read.format("hudi").load(TABLE_PATH).createOrReplaceTempView(TABLE_NAME)

spark.sql(f'''
SELECT
    rider,
    city,
    ts,
    rating,
    _hoodie_commit_time
FROM {TABLE_NAME}
ORDER BY rider
''').show(truncate=False)

+-----+----+-------------------+------+-------------------+
|rider|city|ts                 |rating|_hoodie_commit_time|
+-----+----+-------------------+------+-------------------+
|r1   |SFO |2024-01-10 10:00:00|4.9   |20260425195548307  |
|r2   |SFO |2024-01-10 10:05:00|4.7   |20260425195548307  |
|r3   |NYC |2024-01-10 10:10:00|4.5   |20260425195548307  |
|r4   |LAX |2024-01-10 10:15:00|4.8   |20260425195548307  |
|r5   |SEA |2024-01-10 10:20:00|4.3   |20260425195548307  |
+-----+----+-------------------+------+-------------------+



## Step 8 — Query optimization check with EXPLAIN

The physical plan helps verify partition pruning, pushed filters, and file scan behavior.

For a large production table, this is one of the first checks when a query is unexpectedly slow.

In [8]:
query_df = spark.sql(f'''
SELECT rider, city, rating
FROM {TABLE_NAME}
WHERE city = 'SFO'
''')

query_df.explain(True)
query_df.show(truncate=False)

== Parsed Logical Plan ==
'Project ['rider, 'city, 'rating]
+- 'Filter ('city = SFO)
   +- 'UnresolvedRelation [riders_production_monitoring], [], false

== Analyzed Logical Plan ==
rider: string, city: string, rating: double
Project [rider#27, city#30, rating#29]
+- Filter (city#30 = SFO)
   +- SubqueryAlias riders_production_monitoring
      +- View (`riders_production_monitoring`, [_hoodie_commit_time#22, _hoodie_commit_seqno#23, _hoodie_record_key#24, _hoodie_partition_path#25, _hoodie_file_name#26, rider#27, city#30, ts#28, rating#29])
         +- Project [_hoodie_commit_time#22, _hoodie_commit_seqno#23, _hoodie_record_key#24, _hoodie_partition_path#25, _hoodie_file_name#26, rider#27, city#30, ts#28, rating#29]
            +- Relation [_hoodie_commit_time#22,_hoodie_commit_seqno#23,_hoodie_record_key#24,_hoodie_partition_path#25,_hoodie_file_name#26,rider#27,ts#28,rating#29,city#30] HudiFileGroup

== Optimized Logical Plan ==
Project [rider#27, city#30, rating#29]
+- Filter (isnot

## Step 9 — Write additional commits for monitoring

We create a few updates so the table has commit activity to inspect.

In [9]:
import time

updates = [
    [("r1", "SFO", "2024-01-10 11:00:00", 5.0)],
    [("r6", "DEN", "2024-01-10 11:05:00", 4.4)],
    [("r2", "SFO", "2024-01-10 11:10:00", 4.95)],
]

for batch in updates:
    batch_df = spark.createDataFrame(batch, ["rider", "city", "ts", "rating"]).withColumn("ts", to_timestamp("ts"))

    (
        batch_df.write
        .format("hudi")
        .options(**hudi_write_options)
        .mode("append")
        .save(TABLE_PATH)
    )

    print(f"Written batch: {batch}")
    time.sleep(1)

latest_df = spark.read.format("hudi").load(TABLE_PATH)
latest_df.createOrReplaceTempView(TABLE_NAME)

latest_df.select(
    "_hoodie_commit_time",
    "_hoodie_record_key",
    "rider",
    "city",
    "ts",
    "rating"
).orderBy("rider").show(truncate=False)

26/04/25 19:56:17 WARN ConfigUtils: The configuration key 'hoodie.cleaner.policy' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.policy' instead.
26/04/25 19:56:17 WARN ConfigUtils: The configuration key 'hoodie.cleaner.policy' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.policy' instead.
26/04/25 19:56:17 WARN ConfigUtils: The configuration key 'hoodie.cleaner.commits.retained' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.commits.retained' instead.
26/04/25 19:56:18 WARN ConfigUtils: The configuration key 'hoodie.cleaner.policy' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.policy' instead.
26/04/25 19:56:18 WARN ConfigUtils: The configuration key 'hoodie.cleaner.commits.retained' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.commits.retained' instead.
26/04/25 1

26/04/25 19:56:24 WARN ConfigUtils: The configuration key 'hoodie.cleaner.policy' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.policy' instead.
26/04/25 19:56:24 WARN ConfigUtils: The configuration key 'hoodie.cleaner.commits.retained' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.commits.retained' instead.


26/04/25 19:56:30 WARN ConfigUtils: The configuration key 'hoodie.clean.async' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.async.enabled' instead.
26/04/25 19:56:31 WARN ConfigUtils: The configuration key 'hoodie.cleaner.policy' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.policy' instead.
26/04/25 19:56:31 WARN ConfigUtils: The configuration key 'hoodie.cleaner.commits.retained' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.commits.retained' instead.
26/04/25 19:56:31 WARN ConfigUtils: The configuration key 'hoodie.cleaner.policy' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.policy' instead.
26/04/25 19:56:31 WARN ConfigUtils: The configuration key 'hoodie.cleaner.policy' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.policy' instead.
26/04/25 19:56:31 WARN Con

[Stage 72:>                                                         (0 + 1) / 1]

26/04/25 19:56:39 WARN HoodieTableFileSystemView: Partition: DEN is not available in store


26/04/25 19:56:40 WARN HoodieTableFileSystemView: Partition: DEN is not available in store
26/04/25 19:56:40 WARN ConfigUtils: The configuration key 'hoodie.cleaner.policy' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.policy' instead.
26/04/25 19:56:40 WARN ConfigUtils: The configuration key 'hoodie.cleaner.commits.retained' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.commits.retained' instead.
26/04/25 19:56:44 WARN HoodieTableFileSystemView: Partition: DEN is not available in store


26/04/25 19:56:47 WARN ConfigUtils: The configuration key 'hoodie.clean.async' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.async.enabled' instead.
26/04/25 19:56:49 WARN ConfigUtils: The configuration key 'hoodie.cleaner.policy' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.policy' instead.
26/04/25 19:56:49 WARN ConfigUtils: The configuration key 'hoodie.cleaner.commits.retained' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.commits.retained' instead.
26/04/25 19:56:49 WARN ConfigUtils: The configuration key 'hoodie.cleaner.policy' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.policy' instead.
26/04/25 19:56:49 WARN ConfigUtils: The configuration key 'hoodie.cleaner.policy' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.policy' instead.
26/04/25 19:56:49 WARN Con

26/04/25 19:56:59 WARN ConfigUtils: The configuration key 'hoodie.cleaner.policy' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.policy' instead.
26/04/25 19:56:59 WARN ConfigUtils: The configuration key 'hoodie.cleaner.commits.retained' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.commits.retained' instead.


26/04/25 19:57:07 WARN ConfigUtils: The configuration key 'hoodie.clean.async' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.async.enabled' instead.
26/04/25 19:57:08 WARN ConfigUtils: The configuration key 'hoodie.cleaner.policy' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.policy' instead.
26/04/25 19:57:08 WARN ConfigUtils: The configuration key 'hoodie.cleaner.commits.retained' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.commits.retained' instead.
26/04/25 19:57:08 WARN ConfigUtils: The configuration key 'hoodie.cleaner.policy' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.policy' instead.
26/04/25 19:57:10 WARN ConfigUtils: The configuration key 'hoodie.cleaner.policy' has been deprecated and may be removed in the future. Please use the new key 'hoodie.clean.policy' instead.
26/04/25 19:57:11 WARN Con

## Step 10 — Inspect commit activity from metadata columns

For quick notebook diagnostics, `_hoodie_commit_time` is a simple way to see which commits produced the current records.

In [10]:
commits = [
    row["_hoodie_commit_time"]
    for row in latest_df
        .select("_hoodie_commit_time")
        .distinct()
        .orderBy("_hoodie_commit_time")
        .collect()
]

print("Visible commit instants in current table snapshot:")
for commit in commits:
    print(commit)

print("\nNumber of visible commit instants:", len(commits))

Visible commit instants in current table snapshot:
20260425195548307
20260425195617768
20260425195633499
20260425195651056

Number of visible commit instants: 4


## Step 11 — Inspect table files and metadata folders

This helps debug partition layout, file counts, and metadata table presence.

In [11]:
from pathlib import Path

table_path = Path(TABLE_PATH)

print("Top-level table entries:")
for path in sorted(table_path.iterdir()):
    print(path.name)

print("\nSample files:")
files = [p for p in sorted(table_path.rglob("*")) if p.is_file()]
for path in files[:100]:
    print(path.relative_to(table_path))

if len(files) > 100:
    print(f"... {len(files) - 100} more files")

Top-level table entries:
.hoodie
DEN
LAX
NYC
SEA
SFO

Sample files:
.hoodie/.hoodie.properties.crc
.hoodie/.index_defs/.index.json.crc
.hoodie/.index_defs/index.json
.hoodie/hoodie.properties
.hoodie/metadata/.hoodie/.hoodie.properties.crc
.hoodie/metadata/.hoodie/hoodie.properties
.hoodie/metadata/.hoodie/timeline/.00000000000000000.deltacommit.inflight.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000000.deltacommit.requested.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000000_20260425195551650.deltacommit.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000001.deltacommit.inflight.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000001.deltacommit.requested.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000001_20260425195554183.deltacommit.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000002.deltacommit.inflight.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000002.deltacommit.requested.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000002_20260425195558121.d

## Step 12 — Basic file count by partition

Too many small files can hurt query performance. This simple inspection shows how many files each partition contains.

In [12]:
from collections import defaultdict

partition_file_counts = defaultdict(int)

for file_path in table_path.rglob("*.parquet"):
    partition = file_path.parent.relative_to(table_path)
    partition_file_counts[str(partition)] += 1

print("Parquet file count by partition:")
for partition, count in sorted(partition_file_counts.items()):
    print(f"{partition}: {count}")

Parquet file count by partition:
DEN: 1
LAX: 1
NYC: 1
SEA: 1
SFO: 3


## Step 13 — Cleaner and retention notes

Hudi cleaning removes obsolete file versions according to table cleaner policy.

For this notebook, cleaning is configured with:

- `hoodie.cleaner.policy = KEEP_LATEST_COMMITS`
- `hoodie.cleaner.commits.retained = 3`

In production, retention must be aligned with:

- Time travel requirements
- Rollback requirements
- Incremental reader checkpoints
- Downstream recovery windows

In [13]:
print("Cleaner-related options:")
for key in [
    "hoodie.clean.automatic",
    "hoodie.clean.async",
    "hoodie.cleaner.policy",
    "hoodie.cleaner.commits.retained",
]:
    print(f"{key} = {hudi_write_options.get(key)}")

Cleaner-related options:
hoodie.clean.automatic = true
hoodie.clean.async = false
hoodie.cleaner.policy = KEEP_LATEST_COMMITS
hoodie.cleaner.commits.retained = 3


## Step 14 — Debugging checklist

Use this checklist when troubleshooting Hudi jobs:

1. Confirm the Hudi bundle version matches your Spark and Scala version.
2. Check that `spark.serializer` is Kryo.
3. Confirm record key, partition path, and precombine field are correct.
4. Check whether the table is COW or MOR.
5. Inspect `_hoodie_commit_time` to verify write activity.
6. Use `explain(True)` for slow queries.
7. Check small file counts per partition.
8. Verify cleaner retention does not remove files needed by incremental readers.
9. For streaming, verify checkpoint path and query progress.
10. For MOR, verify compaction scheduling and execution.

## Step 15 — Scaling guidelines

Practical production guidelines:

- Use a stable record key with low collision risk.
- Choose partition columns based on query filters and data volume.
- Avoid over-partitioning into tiny folders.
- Tune `spark.sql.shuffle.partitions` based on input size and cluster capacity.
- Enable AQE for mixed-size workloads.
- Use Hudi metadata table for large datasets.
- Monitor commit duration, file counts, and failed writes.
- Keep enough cleaner retention for incremental consumers.
- Use separate checkpoint directories for independent streaming jobs.

## Step 16 — Summary

In this notebook, you reviewed production-oriented Hudi configuration, query inspection, commit monitoring, and table maintenance concepts.

Key takeaways:

- Production Hudi jobs need explicit Spark and Hudi configuration.
- Metadata indexing and AQE can improve query performance.
- Commit metadata is useful for monitoring and debugging.
- Cleaning and retention settings must match your recovery and incremental-read requirements.
- File layout and partition strategy have a major impact on performance.